In [1]:
import os
import math

import torch
from torch.utils.data import DataLoader

from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    default_data_collator
)

from peft import PeftModel

c:\SQLWhisper\model-tuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 09:37:36.501000 36540 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1106 09:37:37.272000 36540 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, ##quantizaiton in 4 btis
    bnb_4bit_quant_type="nf4",## ahndels rpecisison there ar 16 possible numebrs in 4 btis sicne bits are clsoer to 0 nf4 keeps the 16 valeus clsoes to 0
    bnb_4bit_compute_dtype=torch.bfloat16 ## 16 computations bits
    
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto",
    trust_remote_code = True
).eval()


tokenizer = AutoTokenizer.from_pretrained(
    "../models/qwen-lora-tuned-sql"
)

Temp_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto",
    trust_remote_code = True
).eval()


tuned_model = PeftModel.from_pretrained(
    
    Temp_model,
    "../models/qwen-lora-tuned-sql"
    
).eval()










Loading weights: 100%|██████████| 338/338 [00:05<00:00, 59.66it/s]


In [3]:
def tokenize(batch):
    texts = []

    for instruction, out in zip(batch["instruction"], batch["response"]):
        text = f"Instruction: {instruction}\nResponse: {out}"
        texts.append(text)

    tokens = tokenizer(
    texts,
    padding="max_length",
    max_length=1024,
    truncation=True,
    return_tensors="pt"
    ) 

    tokens["labels"] = tokens["input_ids"].clone()
    tokens["labels"][tokens["attention_mask"] == 0] = -100

    return tokens


In [4]:
eval_ds = load_dataset(
    "json",
    data_files="test.jsonl",
    split="train"
)
eval_ds = eval_ds.map(
    tokenize,
    batched=True,
    remove_columns=eval_ds.column_names
)

Generating train split: 160 examples [00:00, 11436.80 examples/s]
Map: 100%|██████████| 160/160 [00:00<00:00, 1389.14 examples/s]


In [5]:
eval_loader = DataLoader(
    eval_ds,
    batch_size=1,
    collate_fn=default_data_collator
)

In [6]:
with torch.no_grad():
    @torch.no_grad()
    def compute_perplexity(model):
        losses = []

        for batch in eval_loader:

            for key in batch:
                batch[key] = batch[key].to("cuda")

            loss = model(**batch).loss
            losses.append(loss.item())

        return math.exp(sum(losses) / len(losses))

In [7]:
print(f'Base Model Perplexity: {compute_perplexity(model):.2f}')
print(f'Tuned Model Perplexity: {compute_perplexity(tuned_model):.2f}')

Base Model Perplexity: 5.19
Tuned Model Perplexity: 1.98


In [8]:
import torch

print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved() / 1024**3, "GB")
print("Total:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

Allocated: 2.2790489196777344 GB
Reserved: 3.86328125 GB
Total: 5.99951171875 GB


In [12]:
import torch



raw_data =  load_dataset(
    "json",
    data_files="test.jsonl",
    split="train"
)

refs = raw_data["response"]

def generate_sql(model, instruction):
    model.eval()

    prompt = f"Instruction: {instruction}\nResponse:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [13]:
raw_data['instruction'][5]

'Translate this question into ONE read-only PostgreSQL 14+ SELECT query. Use only the schema below. Return SQL only; no markdown or explanation.\n\nDatabase schema:\nCREATE TABLE patients (patient_id BIGINT PRIMARY KEY, patient_name TEXT NOT NULL, municipality TEXT, created_on DATE NOT NULL);\nCREATE TABLE services (service_id BIGINT PRIMARY KEY, service_name TEXT NOT NULL, specialty TEXT, standard_fee NUMERIC(12,2), offered BOOLEAN NOT NULL);\nCREATE TABLE appointments (appointment_id BIGINT PRIMARY KEY, patient_id BIGINT NOT NULL REFERENCES patients(patient_id), service_id BIGINT NOT NULL REFERENCES services(service_id), scheduled_on DATE NOT NULL, duration_minutes INTEGER NOT NULL, billed_amount NUMERIC(12,2), appointment_status TEXT NOT NULL);\nCREATE TABLE inquiries (inquiry_id BIGINT PRIMARY KEY, patient_id BIGINT NOT NULL REFERENCES patients(patient_id), submitted_on DATE NOT NULL, status TEXT NOT NULL);\n\nQuestion: Which patients have at least 6 appointments records?'

In [ ]:
print(generate_sql(tuned_model,raw_data['instruction'][5]))

In [ ]:
print(generate_sql(model,raw_data['instruction'][5]))